# De una URL a un corpus, con criterio

Material complementario · Pair programming con inteligencia artificial

---

Vas a construir tu propio corpus de artículos periodísticos con `trafilatura`, la misma herramienta de la Unidad 2. La diferencia con esa clase es cómo vas a trabajar: acompañado por un asistente de IA (ChatGPT, Gemini, Copilot, un agente de código, el que prefieras), pero bajo una regla que no es técnica sino metodológica.

**La regla en una frase:** la IA puede traducir una intención tuya al lenguaje de la computadora, y puede automatizar lo repetitivo. Pero definir el problema —qué corpus, para qué pregunta, con qué criterio— y evaluar si el resultado sirve, es trabajo exclusivamente tuyo. Eso no se delega.

Este material tiene una doble intención:

1. **Nivel herramienta.** Usar IA para resolver obstáculos puntuales: una sintaxis que no recordás, un error que no entendés, una función que no sabés cómo se llama.
2. **Nivel propulsor.** Usar IA para ampliar lo que sos capaz de imaginar o construir: una alternativa de diseño que no se te había ocurrido, una forma de mirar el problema que cambia tu pregunta.

Vas a encontrar ambos niveles mezclados en el camino. Parte del ejercicio es aprender a distinguirlos.

Tres ideas de la cátedra sostienen todo esto, y las vas a poner a prueba en este mismo notebook:

- **Perspectiva del analista.** No formamos desarrolladores de software. El objetivo es pensamiento computacional: estructurar el problema, y entender lo bastante del lenguaje y las librerías como para poder leer, ajustar y reescribir lo que la IA te proponga.
- **Herramientas que median la cognición.** Python, el entorno, la computadora —y ahora la IA— no son un fin en sí mismos. Son medios que amplían (o restringen) tu forma de pensar el problema.
- **Asistencia, no delegación cognitiva.** Pedirle ayuda a un modelo de lenguaje no es delegarle el pensamiento. Es una conversación: das un paso, lo analizás, reformulás. Vos definís la tarea, indagás el sentido, y verificás la respuesta con tu propio criterio.

---

## Antes de empezar: armá el entorno

Ya hiciste este procedimiento para la Unidad 2 (`002_web_scraping`). Acá lo repetís desde cero, en una carpeta nueva —repetirlo una vez más es parte de que quede incorporado como rutina, no como receta memorizada para una sola vez.

1. Descargá esta carpeta completa (`pair_programming_ia/`, con este notebook, `requirements.txt` y `datos/stopwords_es.txt`) y ubicala en una ruta **sin espacios ni acentos** (por ejemplo `C:\lab-pln\pair_programming_ia` o `~/lab-pln/pair_programming_ia`).
2. Abrí una terminal **parado dentro de esa carpeta**.
3. Creá el entorno e instalá las librerías:

   ```
   uv venv --python 3.12
   uv pip install -r requirements.txt
   ```

4. Abrí la carpeta en VS Code (`Archivo → Abrir carpeta`, la carpeta, no un archivo suelto) y abrí este notebook.
5. Arriba a la derecha, en *Select Kernel*, elegí el que dice `.venv`.

Si algo falla, es el mismo tipo de error que ya viste en la Unidad 2. El `ANTES_DE_LA_CLASE.md` de esa unidad tiene la tabla de "si algo falla" completa.

**No le pidas a la IA que te instale el entorno por vos.** Podés preguntarle qué significa un mensaje de error puntual, pero correr estos comandos y verificar que funcionan es la parte que te toca hacer a mano.

---

## 0. Antes de tocar el teclado: elegir qué vas a bajar

`trafilatura` va a resolver un problema técnico por vos: decidir qué parte de un HTML es "el artículo" y qué parte es ruido (menú, publicidad, notas relacionadas). Es un problema real y no trivial, y la herramienta lo resuelve bien.

Pero hay un problema que ninguna herramienta resuelve por vos: **qué sitio elegís y por qué.** Esa decisión es enteramente tuya, y es anterior a cualquier línea de código.

Respondé esto en la celda de abajo, antes de escribir código. Podés conversarlo con tu asistente de IA para pensarlo en voz alta, pero la respuesta final —y su justificación— tiene que ser tuya:

- ¿Qué pregunta sobre el mundo te gustaría que este corpus te ayude a responder? Elegí el sitio en función de esa pregunta —no elijas primero un sitio "porque se deja scrapear fácil" y después inventes la pregunta para justificarlo.
- Si un compañero o compañera eligiera un sitio distinto para estudiar el mismo tema, ¿en qué se parecerían sus corpus y en qué serían irreconciliables entre sí? ¿Qué te dice eso sobre la idea de que "los datos hablan solos"?
- ¿Qué vas a poder decir con este corpus aunque el código corra sin errores, y qué no vas a poder decir de ninguna manera? (Un corpus de un solo medio, ¿te permite hablar de "lo que dice la prensa" en general?)

Elegí, para lo que sigue, **un sitio de noticias o blog en español** y **una URL de una nota puntual** de ese sitio.

**Tu respuesta (Momento 0):**

_Escribí acá tu sitio elegido, la pregunta que te interesa, y tu justificación._

In [ ]:
import time
from pathlib import Path

import requests

CABECERAS = {
    "User-Agent": "LabPLN 2026 - materia de PLN (poné-tu-mail-acá@ejemplo.com)"
}

DATOS = Path("datos")
DATOS.mkdir(exist_ok=True)


def pedir(url, pausa=1.0):
    """Descarga una página portándose bien."""
    respuesta = requests.get(url, headers=CABECERAS, timeout=20)
    respuesta.raise_for_status()
    time.sleep(pausa)
    return respuesta.text


print("Listo.")

In [ ]:
# Pegá acá la URL que elegiste en el Momento 0.
URL = ""  # ej: "https://ejemplo.com/una-nota"

html = pedir(URL)
print(f"Descargados {len(html)} caracteres de HTML.")

---

## 1. Trafilatura extrae, BeautifulSoup te muestra cuánto quedó afuera

`trafilatura` fue hecha para una sola cosa: dado el HTML de una nota, devolver el cuerpo de la nota, descartando menú, publicidad, notas relacionadas y pie de página. No es magia: son reglas afinadas contra miles de sitios reales. Es exactamente el trabajo que decidimos no hacer a mano.

Le pasamos el HTML que ya bajamos con nuestra función `pedir()`, para mantener una sola forma de pedir páginas: con nuestras cabeceras y nuestra pausa.

In [ ]:
import trafilatura

texto = trafilatura.extract(html)
print(texto[:600] if texto else "trafilatura no encontró texto de artículo en esta URL.")

In [ ]:
from bs4 import BeautifulSoup

sopa = BeautifulSoup(html, "html.parser")

print("BeautifulSoup get_text():", len(sopa.get_text()), "caracteres")
print("trafilatura extract():   ", len(texto) if texto else 0, "caracteres")

La diferencia entre ambos números es lo que trafilatura decidió que no era "el artículo". En un corpus, todo eso sería ruido que contarías como si fuera lenguaje de la nota.

Pensá y respondé:

- Ese porcentaje descartado, ¿era irrelevante para la pregunta que definiste en el Momento 0? ¿O hay preguntas de investigación donde justamente lo descartado (comentarios de lectores, volanta, sección del diario) sería el dato que te interesa?
- Pedile a tu asistente de IA que te explique, en general, qué tipo de reglas usa una herramienta como trafilatura para decidir qué es "el cuerpo del artículo". ¿Esas reglas fueron pensadas para tu sitio en particular, o para sitios en general? ¿Qué consecuencia tiene eso para vos?
- **(Nivel propulsor)** ¿Qué te permite hacer esta herramienta que vos, en el tiempo de esta clase, jamás harías a mano? ¿Y qué te vuelve menos capaz de notar, precisamente porque lo hace tan bien y tan rápido?

**Tu respuesta (Momento 1):**

---

## 2. Un registro con procedencia

Un archivo con texto suelto no es un corpus. Es una carpeta.

Un corpus permite responder: ¿de dónde salió esto?, ¿quién lo escribió?, ¿cuándo se publicó?, ¿cuándo lo capturé yo? Sin esas respuestas no podés citar, no podés comparar por fecha, y no podés reproducir tu propio trabajo dentro de seis meses.

Trafilatura ya extrae todo eso —solo hay que pedírselo. Le pasamos también la `url`: trafilatura recibe el HTML suelto y no tiene manera de adivinar de dónde salió. Ese dato lo tenemos nosotros, que lo descargamos.

In [ ]:
import json

crudo = trafilatura.extract(
    html, url=URL, output_format="json", with_metadata=True
)
datos_articulo = json.loads(crudo)

print("Título: ", datos_articulo["title"])
print("Autoría:", datos_articulo["author"])
print("Fecha:  ", datos_articulo["date"])
print("Medio:  ", datos_articulo["hostname"])

Alguno de estos campos puede venir vacío (`None`): no todos los sitios marcan la autoría o la fecha de forma que trafilatura pueda reconocerla. Eso también es información —sobre el sitio, no un error tuyo.

Esto conecta con la pregunta que sostiene toda la materia: **¿qué se pierde en cada paso, y qué queda justificado afirmar con lo que quedó?** Un campo de metadata vacío es, literalmente, algo que se perdió en el paso de extracción.

---

## 3. Capturar sin romper: el error no es un dato

Vas a necesitar más de una URL para armar un corpus. Cuando trabajás en lote, algo va a fallar: una URL rota, un artículo que trafilatura no puede leer, un sitio caído. Esta función devuelve un registro **o** un motivo de falla —nunca los mezcla.

In [ ]:
from datetime import date


def capturar(url):
    """Devuelve (registro, estado). Si algo falla, el registro es None."""
    try:
        html_pagina = pedir(url)
    except Exception as error:
        return None, f"no se pudo descargar: {error}"

    crudo = trafilatura.extract(
        html_pagina, url=url, output_format="json", with_metadata=True
    )
    if crudo is None:
        return None, "no se encontró texto de artículo"

    datos_extraidos = json.loads(crudo)
    registro = {
        "url": url,
        "medio": datos_extraidos["hostname"],
        "titulo": datos_extraidos["title"],
        "autoria": datos_extraidos["author"],
        "fecha_publicacion": datos_extraidos["date"],
        "fecha_captura": date.today().isoformat(),
        "texto": datos_extraidos["text"],
    }
    return registro, "ok"


print("Función lista.")

Fijate que si algo falla, la función devuelve `None` y un string explicando el motivo —nunca devuelve ese string como si fuera el texto del artículo. Es tentador escribir `return "No se pudo extraer el texto"` cuando algo falla: es una trampa, porque ese mensaje entraría a la columna de texto y sería indistinguible de un artículo real. Cuando contés palabras más adelante, "no", "se" y "pudo" aparecerían entre las más frecuentes de tu corpus, y no entenderías por qué.

Pensá y respondé:

- Guardamos `fecha_captura` además de `fecha_publicacion`. Si repetís este mismo scraping dentro de seis meses, ¿qué pudo haber cambiado en el sitio sin que lo notes? ¿De qué error de interpretación te protege esa segunda fecha?
- Un sitio puede devolver una página de "no encontrado" con código de estado 200 (éxito) en vez de 404. Si eso pasara y vos no revisaras el contenido del registro —solo confiaras en que `estado == "ok"`— ¿qué texto entraría a tu corpus haciéndose pasar por un artículo?
- De los pasos que diste hasta acá (elegir sitio, pedir la página, extraer texto, armar el registro), marcá cuáles fueron una decisión exclusivamente tuya y cuáles correctamente delegaste en la herramienta por ser mecánicos. ¿Dónde está la línea?

**Tu respuesta (Momento 2):**

---

## 4. En lote: tu propio corpus

Armá una lista de entre 4 y 6 URLs. Pueden ser del mismo sitio que elegiste en el Momento 0, o de sitios distintos si tu pregunta de investigación lo pide —eso también es una decisión tuya, no una de la herramienta.

**Sugerencia:** incluí a propósito una URL que sepas que va a fallar (mal escrita, o de una nota que ya no existe). Vas a necesitar ver cómo se comporta tu corpus cuando algo sale mal, no solo cuando todo sale bien.

In [ ]:
URLS = [
    # Completá con tus propias URLs.
]

corpus = []
fallas = []

for url in URLS:
    registro, estado = capturar(url)
    if estado == "ok":
        corpus.append(registro)
        print("OK     ", registro["titulo"][:55])
    else:
        fallas.append({"url": url, "motivo": estado})
        print("FALLÓ  ", url)
        print("        →", estado)

print()
print("Corpus:", len(corpus), "artículos.  Fallas:", len(fallas))

---

## 5. Guardarlo como corpus (JSONL)

Podríamos usar CSV, pero los textos largos tienen comas, comillas y saltos de línea adentro, y el CSV se rompe justo cuando el corpus se pone interesante.

**JSONL** es un archivo de texto con un registro por línea. No hay que escapar nada, se puede ir agregando de a uno, y pandas lo lee en una línea.

In [ ]:
ARCHIVO_CORPUS = DATOS / "mi_corpus.jsonl"

with open(ARCHIVO_CORPUS, "w", encoding="utf-8") as archivo:
    for registro in corpus:
        archivo.write(json.dumps(registro, ensure_ascii=False) + "\n")

print("Guardado en:", ARCHIVO_CORPUS)

In [ ]:
import pandas as pd

df = pd.read_json(ARCHIVO_CORPUS, lines=True)
df[["medio", "fecha_publicacion", "autoria", "titulo"]]

---

## 6. Dos limpiezas que no se mezclan

| | Limpieza de captura | Preprocesamiento de análisis |
|---|---|---|
| Qué hace | Normaliza espacios y saltos | Minúsculas, saca puntuación y palabras vacías |
| Cuándo | Una vez, al capturar | Cada vez que analizás |
| ¿Se guarda? | **Sí**, es el corpus | **No**, se recalcula |
| Cómo es | Conservadora, reversible | Destructiva, descartable |

La regla: **el corpus se guarda lo más parecido posible al original.** Todo lo que destruye información se hace después, para poder rehacerlo distinto.

In [ ]:
import re


def normalizar(texto):
    """Limpieza de captura: ordena espacios. No borra información."""
    texto = texto.replace("\xa0", " ")
    texto = re.sub(r"[ \t]+", " ", texto)
    texto = re.sub(r"\n{3,}", "\n\n", texto)
    return texto.strip()


print(repr(normalizar("Hola\xa0\xa0mundo   con    espacios.")))

In [ ]:
STOPWORDS = set(
    (DATOS / "stopwords_es.txt").read_text(encoding="utf-8").split()
)


def palabras_significativas(texto):
    """Preprocesamiento de análisis: destructivo. El resultado NO se guarda."""
    texto = texto.lower()
    texto = re.sub(r"[^\w\s]", " ", texto)
    return [
        palabra
        for palabra in texto.split()
        if palabra not in STOPWORDS and len(palabra) > 2
    ]


print("Palabras vacías cargadas:", len(STOPWORDS))

In [ ]:
from collections import Counter

todo_el_corpus = " ".join(df["texto"])
frecuencias = Counter(palabras_significativas(todo_el_corpus))

for palabra, veces in frecuencias.most_common(15):
    print(f"{veces:4}  {palabra}")

In [ ]:
import matplotlib.pyplot as plt

top = frecuencias.most_common(15)
palabras = [palabra for palabra, _ in top][::-1]
veces = [cantidad for _, cantidad in top][::-1]

plt.figure(figsize=(8, 5))
plt.barh(palabras, veces)
plt.title("Palabras más frecuentes de tu corpus")
plt.xlabel("apariciones")
plt.tight_layout()
plt.show()

Contar frecuencias asume que "la palabra que más se repite" es la interesante. No siempre es cierto.

Pensá y respondé, mirando tu propio gráfico:

- Buscá en tu lista de las 15 palabras más frecuentes una que no te diga nada sobre tu pregunta de investigación, y pensá si hay una palabra rara (que no aparece en el top 15) que sea más reveladora que cualquiera de las quince.
- `stopwords_es.txt` es una lista genérica de palabras vacías en español. Para tu corpus específico, ¿hay alguna palabra muy frecuente que **no** deberías descartar porque es justamente tu objeto de estudio? ¿Cómo lo sabés vos, y por qué la lista genérica no podía saberlo?
- Pedile a tu asistente de IA que te sugiera una forma distinta de medir "de qué habla" tu corpus, más allá de contar frecuencias (por ejemplo, bigramas, o palabras que aparecen juntas). Evaluá la sugerencia: ¿es mejor para tu pregunta específica, o es solo una técnica más sofisticada que no te acerca a lo que querías saber?

**Tu respuesta (Momento 3):**

---

## 7. Pair programming: separar lo tuyo de lo delegado

Repasá todo el camino que hiciste: elegir el sitio, pedir la página, extraer con trafilatura, armar el registro, capturar en lote, guardar el corpus, elegir qué stopwords sacar.

Escribí una bitácora breve (no hace falta que sea larga) con estos tres puntos:

1. **Un momento en que usaste la IA como herramienta**: le pediste algo que ya sabías hacer, o casi, solo para ganar tiempo o resolver una sintaxis puntual.
2. **Un momento en que la usaste como propulsor**: la respuesta te mostró una posibilidad, un enfoque o una forma de resolver algo que no se te había ocurrido, y que cambió lo que hiciste después.
3. **Un momento en que la definición del problema o la evaluación del resultado fue exclusivamente tuya**, sin importar qué tan bien se lo hubieras preguntado a la IA —algo que ninguna herramienta podía decidir en tu lugar.

**Tu bitácora:**

---

## Glosario rápido

- **Corpus**: conjunto de textos reunidos con un criterio, junto con su procedencia (de dónde salió cada uno, quién lo escribió, cuándo).
- **Procedencia**: los datos que permiten rastrear el origen de un texto —URL, medio, autoría, fecha de publicación, fecha de captura.
- **Ruido**: en un HTML, todo lo que no es el cuerpo del artículo (menú, publicidad, notas relacionadas).
- **Limpieza de captura**: normalización conservadora que se guarda como parte del corpus.
- **Preprocesamiento de análisis**: transformación destructiva (minúsculas, sin puntuación, sin stopwords) que se recalcula cada vez y nunca se guarda.
- **Herramienta (nivel 1 de uso de IA)**: usar el asistente para resolver algo puntual que ya sabés hacer, para ganar tiempo.
- **Propulsor (nivel 2 de uso de IA)**: usar el asistente para ampliar lo que sos capaz de imaginar, diseñar o construir —sin que eso reemplace tu criterio para definir el problema o evaluar el resultado.

---

## Lo que hiciste hoy

Bajaste un HTML, dejaste que una herramienta decidiera qué era ruido y qué era artículo, armaste un registro con procedencia, capturaste en lote sin confundir errores con datos, y contaste frecuencias sabiendo que esa cuenta no es neutral.

En cada uno de esos pasos hubo una parte que le correspondía a la herramienta —incluida la IA con la que trabajaste— y una parte que te correspondía exclusivamente a vos: definir el problema, elegir qué preguntas valía la pena hacer, y evaluar si lo que obtuviste sirve para lo que querías saber.

La pregunta que sostiene toda la materia no cambia acá: **¿qué se perdió en cada paso, y qué queda justificado afirmar con lo que quedó?**